# Order return prediction

### 1) Setup & Imports

In [ ]:
import os
import random
import csv
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm import tqdm
from itertools import combinations

import sklearn
from sklearn.metrics import accuracy_score

import torch
from torch import nn
from torch_geometric.data import HeteroData, DataLoader
from torch_geometric.datasets import OGB_MAG
from torch_geometric.nn import SAGEConv, to_hetero, GCNConv, GATv2Conv, HANConv, Linear
import torch_geometric.transforms as T
import torch.nn.functional as F


In [ ]:
# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["TORCH_USE_CUDA_DSA"] = '1'

### 2) Data Loading

In [ ]:
task_data = pd.read_csv('task1_data.txt',sep = ',')
train_label = pd.read_csv('task1_train_label.txt',sep = '\t',header = None, names = ['order', 'label'])
valid_label = pd.read_csv('task1_valid_label.txt',sep = '\t', header = None, names = ['order', 'label'])
test_query = pd.read_csv('task1_test_query.txt', header = None, names = ['order'])

In [ ]:
train_data = task_data.merge(train_label, how='inner', on='order')
valid_data = task_data.merge(valid_label, how='inner', on='order')
test_data = task_data.merge(test_query, how='inner', on='order') # no label

In [ ]:
train_data.head()

,order,product,customer,color,size,group,label
0,298957,54654,192219,169,10,11,0
1,298957,57127,192219,611,10,11,0
2,570617,23677,322757,43,24,19,1
3,570617,54993,322757,543,24,19,1
4,654410,53969,305375,313,17,27,2


In [ ]:
valid_data.head()

,order,product,customer,color,size,group,label
0,529565,10260,212452,92,17,19,0
1,529565,51789,212452,169,17,19,0
2,529565,12020,212452,313,17,19,0
3,529565,17961,212452,627,10,19,0
4,529565,47772,212452,409,17,6,0


In [ ]:
test_data.head()

,order,product,customer,color,size,group
0,610749,1908,200460,590,10,19
1,610749,48440,200460,10,10,19
2,610749,35567,200460,313,10,6
3,610749,41103,200460,313,10,6
4,610749,31029,200460,313,10,6


In [ ]:
train_data['label'].value_counts()

label
1    1116464
0     431381
2     345549
Name: count, dtype: int64

1) product_info (once made, just use reader)

In [ ]:
# product_info = defaultdict(int)
# for i in tqdm(range(len(train_data))):
#   row = train_data.iloc[i,:]
#   if product_info[row['product']] == 0:
#     product_info[row['product']] = [row['color'],row['size'],row['group']]
#   else :
#     continue


In [ ]:
# with open('/data/home/yjkim/datamining_task1/New_data/product_info.csv', 'w') as f:
#     writer = csv.writer(f)    
#     for k,v in product_info.items():
#         writer.writerow([k] + v)

In [ ]:
with open('/data/home/yjkim/datamining_task1/New_data/product_info.csv', 'r') as f:
    reader = csv.reader(f)    
    product_info = defaultdict(list)
    for row in reader:
        for i in range(1,len(row)):
            product_info[np.int64(row[0])].append(np.int64(row[i]))

In [ ]:
print(product_info[26196])
print(type(product_info[26196][0]))

[187, 17, 11]
<class 'numpy.int64'>


In [ ]:
p = list(product_info)
p.sort(reverse = True)
print(p[0])

58414


In [ ]:
task_data.shape, train_data.shape, valid_data.shape, test_data.shape

((2666262, 6), (1893394, 7), (386806, 7), (386062, 6))

2. order_product_dict (full)

In [ ]:
order_product_dict = defaultdict(int)
for i in tqdm(range(len(train_data))):
  row = train_data.iloc[i,:]
  if order_product_dict[row['order']] == 0:
    order_product_dict[row['order']] = [[row['product']], row['label']]
  else :
    order_product_dict[row['order']][0].append(row['product'])

  0%|          | 0/1893394 [00:00<?, ?it/s]

100%|██████████| 1893394/1893394 [01:52<00:00, 16843.18it/s]


3. product_num_label

In [ ]:
product_num_label = defaultdict(list)
for order, value in order_product_dict.items():
  product_num_label[len(value[0])].append(value[1])

4. order_product

In [ ]:
order_product = defaultdict(list)
for key,value in product_num_label.items():
  class_num = []
  for i in range(3):
    class_num.append(value.count(i))
  order_product[key] = class_num
s_list = sorted(order_product.items(), key = lambda x : x[0])

5. group_product_dict (reader only)

In [ ]:
# group_product_dict = defaultdict(int)
# for i in tqdm(range(len(train_data))):
#   row = train_data.iloc[i,:]
#   row_label = row['label']
#   if group_product_dict[row['group']] == 0:
#     if row_label == 0:
#       group_product_dict[row['group']] = [1,0,0]
#     elif row_label == 1:
#       group_product_dict[row['group']] = [0,1,0]
#     elif row_label == 2:
#       group_product_dict[row['group']] = [0,0,1]
#   else :
#     if row_label == 0:
#       group_product_dict[row['group']][0] +=1
#     elif row_label == 1:
#       group_product_dict[row['group']][1] +=1
#     elif row_label == 2:
#       group_product_dict[row['group']][2] +=1

100%|██████████| 1893394/1893394 [01:40<00:00, 18765.39it/s]


In [ ]:
# with open('/data/home/yjkim/datamining_task1/New_data/group_product_dict.csv', 'w') as f:
#     writer = csv.writer(f)    
#     for k,v in group_product_dict.items():
#         writer.writerow([k] + v)

In [ ]:
with open('/data/home/yjkim/datamining_task1/New_data/group_product_dict.csv', 'r') as f:
    reader = csv.reader(f)    
    group_product_dict = defaultdict(list)
    for row in reader:
        for i in range(1,len(row)):
            group_product_dict[np.int64(row[0])].append(np.int64(row[i]))

6. size_product_dict (reader only)

In [ ]:
# size_product_dict = defaultdict(int)
# for i in tqdm(range(len(train_data))):
#   row = train_data.iloc[i,:]
#   row_label = row['label']
#   if size_product_dict[row['size']] == 0:
#     if row_label == 0:
#       size_product_dict[row['size']] = [1,0,0]
#     elif row_label == 1:
#       size_product_dict[row['size']] = [0,1,0]
#     elif row_label == 2:
#       size_product_dict[row['size']] = [0,0,1]
#   else :
#     if row_label == 0:
#       size_product_dict[row['size']][0] +=1
#     elif row_label == 1:
#       size_product_dict[row['size']][1] +=1
#     elif row_label == 2:
#       size_product_dict[row['size']][2] +=1

  0%|          | 0/1893394 [00:00<?, ?it/s]

100%|██████████| 1893394/1893394 [01:47<00:00, 17533.66it/s]


In [ ]:
# with open('/data/home/yjkim/datamining_task1/New_data/size_product_dict.csv', 'w') as f:
#     writer = csv.writer(f)    
#     for k,v in size_product_dict.items():
#         writer.writerow([k] + v)

In [ ]:
with open('/data/home/yjkim/datamining_task1/New_data/size_product_dict.csv', 'r') as f:
    reader = csv.reader(f)    
    size_product_dict = defaultdict(list)
    for row in reader:
        for i in range(1,len(row)):
            size_product_dict[np.int64(row[0])].append(np.int64(row[i]))

7. customer_info (reader only)

In [ ]:
# customer - return 관계
customer_info = defaultdict(int)
for i in tqdm(range(len(train_data))):
  row = train_data.iloc[i,:]
  if customer_info[row['customer']] == 0:
    customer_info[row['customer']] = [row['label']]
  else :
    continue

for i in tqdm(range(len(valid_data))):
  row = valid_data.iloc[i,:]
  if customer_info[row['customer']] == 0:
    customer_info[row['customer']] = [row['label']]
  else :
    continue

for i in tqdm(range(len(test_data))):
  row = test_data.iloc[i,:]
  if customer_info[row['customer']] == 0:
    customer_info[row['customer']] = [0] #no label
  else :
    continue


In [ ]:
with open('/data/home/yjkim/datamining_task1/New_data/customer_info.csv', 'w') as f:
    writer = csv.writer(f)    
    for k,v in customer_info.items():
        writer.writerow([k] + v)

In [ ]:
with open('/data/home/yjkim/datamining_task1/New_data/customer_info.csv', 'r') as f:
    reader = csv.reader(f)    
    m = defaultdict(list)
    for row in reader:
        for i in range(1,len(row)):
            m[np.int64(row[0])].append(np.int64(row[i]))

In [ ]:
column_names = ['order','product','customer','color','size','group','label']
corr = train_data[column_names].corr(method='pearson')
corr

,order,product,customer,color,size,group,label
order,1.000000,-0.000561,-0.001507,0.001061,-0.001926,0.000250,0.002247
product,-0.000561,1.000000,-0.000267,0.005640,-0.008649,-0.003215,-0.000678
customer,-0.001507,-0.000267,1.000000,-0.001716,0.001741,0.000054,0.000192
color,0.001061,0.005640,-0.001716,1.000000,0.002162,-0.017047,-0.001024
size,-0.001926,-0.008649,0.001741,0.002162,1.000000,0.033946,-0.008863
group,0.000250,-0.003215,0.000054,-0.017047,0.033946,1.000000,0.001111
label,0.002247,-0.000678,0.000192,-0.001024,-0.008863,0.001111,1.000000


In [ ]:
valid_data.loc[valid_data['order'] == 529565]['product'].tolist()

[10260, 51789, 12020, 17961, 47772]

### 3) Feature Engineering

In [ ]:
product_idx = sorted(list(set(train_data['product'].values.tolist() + valid_data['product'].values.tolist() + test_data['product'].values.tolist())))
product_x = []
for idx in product_idx:
  onehot = [0]*32
  group_info = product_info[idx][2] # group
  onehot[group_info] += 1
  product_x.append(onehot)
#product_x = [product_info[x][1:] for x in product_idx] #size, group

edge_source = [] #order-product
edge_destination = []
# order_edge_source = []#order-order
# order_edge_destination = []
product_edge_source = []#product-product
product_edge_destination = []

order_y = []
order_x = []
# train set에서
for index, row in tqdm(train_data.drop_duplicates(subset = 'order').iterrows()):
  #onehot = [0]*10
  order = row['order']
  label = row['label']
  customer = row['customer']
  p_list = order_product_dict[order][0]
  group_feature = np.zeros(32)

  for p in p_list:
    edge_source.append(order)
    edge_destination.append(p)
    group_feature += np.array(product_x[p])
  combi = combinations(p_list,2)
  for c in combi:
    product_edge_source.append(c[0])
    product_edge_destination.append(c[1])
  # if len(p_list) >= 10:
  #   onehot[9] += 1
  # else:
  #   onehot[len(p_list)-1] += 1
  #onehot.extend(list(np.array(order_product[len(p_list)])/sum(order_product[len(p_list)])))
  group_feature = list(group_feature)
  group_feature.append(len(p_list))
  order_x.append(group_feature)
  if label == 0:
    order_y.append([1,0,0])
  elif label == 1:
    order_y.append([0,1,0])
  elif label == 2:
    order_y.append([0,0,1])

train_num = len(order_y)

# valid set에서
for order_idx in tqdm(valid_data.drop_duplicates(subset = 'order')['order'].tolist()):
  #onehot = [0]*10
  order_df = valid_data.loc[valid_data['order'] == order_idx]
  p_list = order_df['product'].tolist()
  group_feature = np.zeros(32)
  for i, row in order_df.iterrows():
    order = row['order']
    label = row['label']
    product = row['product']
    customer = row['customer']
    group_feature += np.array(product_x[product])
    edge_source.append(order)
    edge_destination.append(product)
  combi = combinations(p_list,2)
  for c in combi:
    product_edge_source.append(c[0])
    product_edge_destination.append(c[1])
  # if len(p_list) >= 10:
  #   onehot[9] += 1
  # else:
  #   onehot[len(p_list)-1] += 1
  # onehot.extend(list(np.array(order_product[len(p_list)])/sum(order_product[len(p_list)])))
  group_feature = list(group_feature)
  group_feature.append(len(p_list))
  order_x.append(group_feature)
  if label == 0:
    order_y.append([1,0,0])
  elif label == 1:
    order_y.append([0,1,0])
  elif label == 2:
    order_y.append([0,0,1])

val_num = len(order_y) - train_num
# test set에서
for order_idx in tqdm(test_data.drop_duplicates(subset = 'order')['order'].tolist()):
  #onehot = [0]*10
  order_df = test_data.loc[test_data['order'] == order_idx]
  p_list = order_df['product'].tolist()
  group_feature = np.zeros(32)
  for i, row in order_df.iterrows():
    order = row['order']
    product = row['product']
    customer = row['customer']
    group_feature += np.array(product_x[product])
    edge_source.append(order)
    edge_destination.append(product)
  combi = combinations(p_list,2)
  for c in combi:
    product_edge_source.append(c[0])
    product_edge_destination.append(c[1])
  # if len(p_list) >= 10:
  #   onehot[9] += 1
  # else:
  #   onehot[len(p_list)-1] += 1
  # onehot.extend(list(np.array(order_product[len(p_list)])/sum(order_product[len(p_list)])))
  group_feature = list(group_feature)
  group_feature.append(len(p_list))
  order_x.append(group_feature)
  order_y.append([0,0,0])

test_num = len(order_y) - train_num - val_num

594430it [00:36, 16275.53it/s]
100%|██████████| 127378/127378 [01:17<00:00, 1639.52it/s]


In [ ]:
customer_ids = list(customer_info.keys())
customer_index_mapping = {cust_id: idx for idx, cust_id in enumerate(customer_ids)}
customer_x = [customer_info[cust_id] for cust_id in customer_ids]

In [ ]:
print(train_num, val_num, test_num)

594430 127377 127378


### 4) Graph Construction

In [ ]:
# data 정의
data = HeteroData()
data['order'].x = torch.tensor(order_x).type(torch.float)
data['product'].x = torch.tensor(product_x).type(torch.float) # feature: color, size, group

data['order'].y = torch.tensor(order_y).type(torch.float)
data['order','contains','product'].edge_index = torch.tensor([edge_source,edge_destination])
data['product','same_order','product'].edge_index = torch.tensor([product_edge_source,product_edge_destination])

data = T.ToUndirected()(data)

data['order'].train_mask = torch.zeros(data['order'].num_nodes, dtype=torch.bool)
data['order'].train_mask[:train_num] = 1
data['order'].val_mask = torch.zeros(data['order'].num_nodes, dtype=torch.bool)
data['order'].val_mask[train_num:train_num+val_num] = 1
data['order'].test_mask = torch.zeros(data['order'].num_nodes, dtype=torch.bool)
data['order'].test_mask[train_num+val_num:] = 1

#data = data.cuda()

In [ ]:
max_index = data['customer','ordered','product'].edge_index.max()
num_nodes = data['customer'].x.size(0)
assert max_index < num_nodes, f"Max index {max_index} exceeds number of nodes {num_nodes}"

In [ ]:
data
#product 의 col size 는 왜 2? color, size, group ?

HeteroData(
  order={
    x=[849185, 33],
    y=[849185, 3],
    train_mask=[849185],
    val_mask=[849185],
    test_mask=[849185],
  },
  product={ x=[58415, 32] },
  (order, contains, product)={ edge_index=[2, 2666262] },
  (product, same_order, product)={ edge_index=[2, 9090776] },
  (product, rev_contains, order)={ edge_index=[2, 2666262] }
)

In [ ]:
torch.cuda.is_available()
torch.cuda.current_device()

0

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

### 5) Model Definition

In [ ]:
class GNN(torch.nn.Module):
    def __init__(self, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), 64)
        self.conv2 = SAGEConv((-1, -1), 128)
        self.conv3 = SAGEConv((-1, -1), 256)
        #self.conv4 = SAGEConv((-1, -1), 256)
        #self.conv5 = SAGEConv((-1, -1), 256)
        self.linear = torch.nn.Linear(256, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        x = self.conv3(x, edge_index).relu()
        #x = self.conv4(x, edge_index)
        #x = self.conv5(x, edge_index)
        x = self.linear(x)
        x = F.softmax(x, dim = -1)
        return x


model = GNN(out_channels=3)
model = to_hetero(model, data.metadata(), aggr='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch.device('cpu')
data, model = data.to(device), model.to(device)
print(device)

cuda


### 6) Training & Testing

In [ ]:
import time
import copy

losses = []
training_accuracies = []
validation_accuracies = []

def test(mask):
  model.eval()
  prediction = []
  pred = model(data.x_dict, data.edge_index_dict)['order'].argmax(dim=-1)[mask].tolist()

  # d_x = data['order'].x.detach().cpu()
  # for i in range(len(pred)):
  #   if d_x[i].item() == 1:
  #     if pred[i][0] >= pred[i][2]:
  #       prediction.append(0)
  #     else:
  #       prediction.append(2)
  #   else:
  #     prediction.append(pred[i].argmax().item())
  answer= data['order'].y.argmax(dim=-1)[mask].tolist()
  acc = accuracy_score(answer,pred)
  return float(acc)

def train_model(model, num_epochs=100):
    """
    model: model to train
    dataloaders: train, val, test data's loader
    criterion: loss function
    optimizer: optimizer to update your model
    """
    since = time.time()
    max_val_acc = 0
    max_val_epoch = 0
    gc.collect()
    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        model.train()            # Set model to training mode

        optimizer.zero_grad()

        out = model(data.x_dict, data.edge_index_dict)
        mask = data['order'].train_mask
        loss = F.cross_entropy(out['order'][mask], data['order'].y[mask])

        loss.backward()
        optimizer.step()

        train_acc = test(data['order'].train_mask)
        val_acc = test(data['order'].val_mask)
        if val_acc > max_val_acc:
          max_val_acc = val_acc
          max_val_epoch = epoch
          torch.save(model, 'winner_best_model.pt')

        losses.append(loss)
        training_accuracies.append(train_acc)
        validation_accuracies.append(val_acc)

        print('Loss: {:.4f}'.format(float(loss)))
        print('Train Accuracy: {:.4f}'.format(float(train_acc)))
        print('Validation Accuracy: {:.4f}'.format(float(val_acc)))

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
    print('best_val_acc: '+str(max_val_acc)+' at epoch: '+str(max_val_epoch))
    #torch.save(model, "/content/drive/MyDrive/graph_mining/" + 'model.pt')



In [ ]:
train_model(model,300)

Epoch 0/299
----------
Loss: 1.0929
Train Accuracy: 0.3949
Validation Accuracy: 0.3875

Epoch 1/299
----------
Loss: 1.0745
Train Accuracy: 0.3948
Validation Accuracy: 0.3874

Epoch 2/299
----------
Loss: 1.0618
Train Accuracy: 0.3986
Validation Accuracy: 0.3909

Epoch 3/299
----------
Loss: 1.0523
Train Accuracy: 0.5597
Validation Accuracy: 0.5544

Epoch 4/299
----------
Loss: 1.0429
Train Accuracy: 0.5719
Validation Accuracy: 0.5649

Epoch 5/299
----------
Loss: 1.0339
Train Accuracy: 0.5616
Validation Accuracy: 0.5545

Epoch 6/299
----------
Loss: 1.0257
Train Accuracy: 0.5677
Validation Accuracy: 0.5605

Epoch 7/299
----------
Loss: 1.0172
Train Accuracy: 0.5755
Validation Accuracy: 0.5682

Epoch 8/299
----------
Loss: 1.0094
Train Accuracy: 0.5763
Validation Accuracy: 0.5692

Epoch 9/299
----------
Loss: 1.0017
Train Accuracy: 0.5731
Validation Accuracy: 0.5658

Epoch 10/299
----------
Loss: 0.9942
Train Accuracy: 0.5714
Validation Accuracy: 0.5642

Epoch 11/299
----------
Loss: 0

KeyboardInterrupt: 

### Other models (HAN & GAT)

In [ ]:
# HAN model

class HAN(nn.Module):
    def __init__(self, dim_in, dim_out, dim_h=128, heads=8):
        super().__init__()
        self.han = HANConv(dim_in, dim_h, heads=heads, dropout=0.6, metadata=data.metadata())
        self.linear = nn.Linear(dim_h, dim_out)
 
    def forward(self, x_dict, edge_index_dict):
        out = self.han(x_dict, edge_index_dict)
        #print(f"Shape of HAN output for 'order': {out['order'].shape}")
        out = self.linear(out['order'])
        return out


model = HAN(dim_in=-1, dim_out=3)
 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data, model = data.to(device), model.to(device)
print(device)

cuda


In [ ]:
import time
import copy

losses = []
training_accuracies = []
validation_accuracies = []

@torch.no_grad() #?
def test(mask):
  model.eval()
  #print(model(data.x_dict, data.edge_index_dict))
  output = model(data.x_dict, data.edge_index_dict)
  pred = output.argmax(dim=-1)

  if mask.dtype == torch.bool:
    masked_pred = pred[mask]  # Correctly apply mask if it's boolean
  else:
    masked_pred = pred[mask.nonzero().squeeze()]

  true_labels = data['order'].y.argmax(dim=-1)  # Make sure to get the class indices if y is not already such
  if mask.dtype == torch.bool:
    masked_labels = true_labels[mask]
  else:
    masked_labels = true_labels[mask.nonzero().squeeze()]

  acc = accuracy_score(masked_labels.tolist(), masked_pred.tolist())
  return float(acc)
 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
max_val_acc = 0
max_val_epoch = 0

for epoch in range(299):
  model.train()
  optimizer.zero_grad()
  out = model(data.x_dict, data.edge_index_dict)
  mask = data['order'].train_mask


  #print("Training output shape:", out.shape)  # Debug print to check output shape
  #print("Mask for training shape and type:", mask.shape, mask.dtype)  # Debug print to check mask shape and type
  
  loss = F.cross_entropy(out[mask], data['order'].y[mask])
  loss.backward()
  optimizer.step()
 
  if epoch % 20 == 0:
    train_acc = test(data['order'].train_mask)
    val_acc = test(data['order'].val_mask)
    if val_acc > max_val_acc:
      max_val_acc = val_acc
      max_val_epoch = epoch
      torch.save(model.state_dict(), 'winner_best_model.pt')
    print(f'Epoch: {epoch:>3} | Train Loss: {loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%')
 
test_acc = test(data['order'].test_mask)
print(f'Test accuracy: {test_acc*100:.2f}%')


print('best_val_acc: '+str(max_val_acc)+' at epoch: '+str(max_val_epoch))
#torch.save(model, "/content/drive/MyDrive/graph_mining/" + 'model.pt')



Epoch:   0 | Train Loss: 1.1088 | Train Acc: 36.22% | Val Acc: 36.70%
Epoch:  20 | Train Loss: 1.0814 | Train Acc: 39.37% | Val Acc: 38.66%
Epoch:  40 | Train Loss: 1.0805 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch:  60 | Train Loss: 1.0801 | Train Acc: 39.47% | Val Acc: 38.74%
Epoch:  80 | Train Loss: 1.0799 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 100 | Train Loss: 1.0797 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 120 | Train Loss: 1.0797 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 140 | Train Loss: 1.0794 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 160 | Train Loss: 1.0794 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 180 | Train Loss: 1.0794 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 200 | Train Loss: 1.0792 | Train Acc: 39.48% | Val Acc: 38.74%


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import precision_score , recall_score , confusion_matrix

print(confusion_matrix(y_true, val_pred))

precision = precision_score(y_true, y_pred,average= "macro")
recall = recall_score(y_true, y_pred,average= "macro")
print('정밀도: {0:.4f}, 재현율: {1:.4f}'.format(precision, recall))

In [ ]:
out = model(data.x_dict, data.edge_index_dict)

In [ ]:
p = out['order'].argmax(dim = -1).tolist()
for i in range(3):
  print(p.count(i))

443918
405267
0


In [ ]:
# GAT model

from torch_geometric.nn import GATConv, Linear, to_hetero

class GAT(torch.nn.Module):
    def __init__(self, dim_h, dim_out):
        super().__init__()
        self.conv = GATConv((-1, -1), dim_h, add_self_loops=False)
        self.linear = nn.Linear(dim_h, dim_out)
 
    def forward(self, x, edge_index):
        h = self.conv(x, edge_index).relu()
        h = self.linear(h)
        return h
    
model = GAT(dim_h=128, dim_out=3)
model = to_hetero(model, data.metadata(), aggr='sum')
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

GraphModule(
  (conv): ModuleDict(
    (order__contains__product): GATConv((-1, -1), 128, heads=1)
    (product__same_order__product): GATConv((-1, -1), 128, heads=1)
    (product__rev_contains__order): GATConv((-1, -1), 128, heads=1)
  )
  (linear): ModuleDict(
    (order): Linear(in_features=128, out_features=3, bias=True)
    (product): Linear(in_features=128, out_features=3, bias=True)
  )
)



def forward(self, x, edge_index):
    x_dict = torch_geometric_nn_to_hetero_transformer_get_dict(x);  x = None
    x__order = x_dict.get('order', None)
    x__product = x_dict.get('product', None);  x_dict = None
    edge_index_dict = torch_geometric_nn_to_hetero_transformer_get_dict(edge_index);  edge_index = None
    edge_index__order__contains__product = edge_index_dict.get(('order', 'contains', 'product'), None)
    edge_index__product__same_order__product = edge_index_dict.get(('product', 'same_order', 'product'), None)
    edge_index__product__rev_contains__order = edge_index_dict.get(

In [ ]:
def test(mask):
    model.eval()
    pred = model(data.x_dict, data.edge_index_dict)['order'].argmax(dim=-1)[mask].tolist()
    answer = data['order'].y.argmax(dim=-1)[mask].tolist()
    acc = accuracy_score(answer,pred)
    return float(acc)
 
for epoch in range(101):
    model.train()
    optimizer.zero_grad()
    out = model(data.x_dict, data.edge_index_dict)['order']
    mask = data['order'].train_mask
    loss = F.cross_entropy(out[mask], data['order'].y[mask])
    loss.backward()
    optimizer.step()
 
    if epoch % 20 == 0:
        train_acc = test(data['order'].train_mask)
        val_acc = test(data['order'].val_mask)
        print(f'Epoch: {epoch:>3} | Train Loss: {loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%')
 
test_acc = test(data['order'].test_mask)
print(f'Test accuracy: {test_acc*100:.2f}%')

Epoch:   0 | Train Loss: 1.0912 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch:  20 | Train Loss: 1.0788 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch:  40 | Train Loss: 1.0785 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch:  60 | Train Loss: 1.0783 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch:  80 | Train Loss: 1.0783 | Train Acc: 39.48% | Val Acc: 38.74%
Epoch: 100 | Train Loss: 1.0782 | Train Acc: 39.48% | Val Acc: 38.74%
Test accuracy: 0.00%
